# 04 — Inference and official evaluation

**One job:** run all detector, crop-model, and full-pipeline comparisons against one frozen C0

Run from top to bottom. Slow actions are off by default.

In [ ]:
%pip install -q ultralytics pycocotools segmentation-models-pytorch

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import json
import sys
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

PROJECT_DIR = Path("/content/drive/MyDrive/Google Colab/Solar Filament Segmentation Challenge")
sys.path.insert(0, str(PROJECT_DIR))
%load_ext autoreload
%autoreload 2

from src.config import DEFAULT_PIPELINE_SETTINGS, TRAINING_SETTINGS
from src.data import copy_raw_data_to_local, load_records, make_split, verify_or_write_split

LOCAL_DATA_DIR = copy_raw_data_to_local(PROJECT_DIR / "data/raw")
TRAIN_IMAGES_DIR = LOCAL_DATA_DIR / "train/train_images"
TEST_IMAGES_DIR = LOCAL_DATA_DIR / "test/test_images"
ANNOTATIONS_PATH = LOCAL_DATA_DIR / "train/MAGFiLO_1.0_Annotations_kaggle2026_train.json"
SPLIT_MANIFEST_PATH = PROJECT_DIR / "data/processed/split_manifest.json"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))

In [ ]:
RUN_C0 = False
RUN_SWEEPS = False
RUN_TTA = False
RUN_ENSEMBLE = False
YOLO_CHECKPOINT = PROJECT_DIR / "models/stage1/yolo26x_filament_detection/weights/best.pt"
UNET_CHECKPOINT = PROJECT_DIR / "models/stage2/crop_unet_b5/best_crop_unet_b5.pt"

## Current best configuration

In [ ]:
from src.experiments import append_registry, load_registry, new_run_dir, save_run_files

registry_path = PROJECT_DIR / "experiments/registry.csv"
registry = load_registry(registry_path)
current_best = registry.sort_values(["kaggle_public", "official_pq"], ascending=False, na_position="last").head(1)
display(current_best[["run_id", "detector", "detector_imgsz", "conf", "nms_iou", "unet_checkpoint", "mask_threshold", "tta", "official_pq", "kaggle_public"]])

## A. Stage 1 in isolation

A1 compares n/s/l/x and image size. A2 sweeps confidence. A3 sweeps NMS IoU and compares end-to-end vs NMS. A4 measures expansion versus >50% crop coverage. A5 tests detector TTA. A6, detector ensembling, is measured end to end in C4. Detector-only cells must report proposals/image, precision, recall, and crop coverage before involving a U-Net.

### A1–A6 experiment record

Use the same question/varies/fixed/cost/result/decision block as Section C. Append detector-only runs to `experiments/stage1/` and the registry.

## B. Stage 2 in isolation

B1 compares oracle Dice/IoU per checkpoint. B2 sweeps the oracle mask threshold. B3 tests 8-way TTA. B4 measures negative-crop rejection at the real 20-pixel bar. B5, U-Net averaging, is measured end to end in C4. Oracle results are diagnostic only.

## C. Full pipeline under the official rule

In [ ]:
from pycocotools import mask as mask_utils
from tqdm.auto import tqdm
from src.inference import predict_instances
from src.metrics import aggregate_official, annotation_to_rle, bootstrap_pq, official_match_counts
from src.models import RotationTTA, load_checkpoint

coco, records = load_records(ANNOTATIONS_PATH, TRAIN_IMAGES_DIR)
_, _, validation_records = verify_or_write_split(records, SPLIT_MANIFEST_PATH)
validation_groups = defaultdict(list)
for record in validation_records:
    validation_groups[record["physical_id"]].append(record)

def evaluate_pipeline(detector, unet, settings, mask_threshold, run_dir, proposals=None):
    rows, predictions, image_counts = [], [], []
    for physical_id, group in tqdm(validation_groups.items()):
        image_proposals = None if proposals is None else proposals[physical_id]
        masks, counts = predict_instances(group[0]["image_path"], detector, unet, device, mask_threshold, settings, proposals=image_proposals)
        rles = [mask_utils.encode(np.asfortranarray(mask.astype(np.uint8))) for mask in masks]
        saved_rles = [{"size": rle["size"], "counts": rle["counts"].decode("ascii")} for rle in rles]
        predictions.append({"physical_id": physical_id, "file_name": group[0]["file_name"], "rles": saved_rles})
        image_counts.append(counts)
        for record in group:
            targets = [annotation_to_rle(annotation, record["height"], record["width"]) for annotation in record["annotations"]]
            rows.append({"physical_id": physical_id, "annotator_image_id": record["annotator_image_id"], **official_match_counts(rles, targets)})
    by_annotator = pd.DataFrame(rows)
    summary = {**aggregate_official(by_annotator), **bootstrap_pq(by_annotator),
               "instances_per_image": float(pd.DataFrame(image_counts)["final_instances"].mean())}
    save_run_files(run_dir, {"pipeline": settings, "mask_threshold": mask_threshold}, predictions, rows, summary)
    return by_annotator, summary

### C0 Reproduce the current best

**Question:** does the reorganized path reproduce the saved current best?

**Varies:** nothing.

**Fixed:** YOLO26x @1280, conf 0.20, NMS IoU 0.20, B5 epoch 13, mask 0.45, crop 1.5, dedup 0.50/0.80, minimum area 20.

**Cost:** ~1 minute on A100 or ~15 minutes on CPU.

**Expected:** PQ 0.4300; TP 786, FP 451, FN 472 in the saved fp16 run.

**Decision:** this remains the control unless a candidate clears uncertainty or earns a leaderboard test.

In [ ]:
if RUN_C0:
    from ultralytics import YOLO

    settings = DEFAULT_PIPELINE_SETTINGS.copy()
    detector = YOLO(str(YOLO_CHECKPOINT))
    unet, checkpoint = load_checkpoint(UNET_CHECKPOINT, "b5", device)
    run_id, run_dir = new_run_dir(PROJECT_DIR / "experiments", "pipeline", "C0", "b5_yolo26x")
    by_annotator_c0, summary_c0 = evaluate_pipeline(detector, unet, settings, 0.45, run_dir)
    append_registry(registry_path, {"run_id": run_id, "timestamp": datetime.now().isoformat(), "notebook": "04_inference_and_evaluation.ipynb",
                                   "section": "C0", "detector": "yolo26x", "detector_imgsz": 1280, "conf": 0.20, "nms_iou": 0.20,
                                   "end2end": False, "unet_checkpoint": UNET_CHECKPOINT.name, "mask_threshold": 0.45, "tta": False,
                                   "official_pq": summary_c0["official_pq"], "sq": summary_c0["official_sq"], "rq": summary_c0["official_rq"],
                                   "tp": summary_c0["official_tp"], "fp": summary_c0["official_fp"], "fn": summary_c0["official_fn"],
                                   "instances_per_image": summary_c0["instances_per_image"], "pq_ci_low": summary_c0["pq_ci_low"],
                                   "pq_ci_high": summary_c0["pq_ci_high"], "device": str(device), "precision": "fp16" if device.type == "cuda" else "fp32"})
    display(pd.Series(summary_c0))
else:
    print("C0 inference skipped. Saved C0: PQ 0.4300496381, SQ 0.6825533378, RQ 0.6300601202, TP 786, FP 451, FN 472.")

### C1.2 NMS IoU sweep

**Question:** does NMS at IoU 0.20 suppress real neighbouring filaments?

**Varies:** `yolo_iou` in {0.20, 0.35, 0.50, 0.70}.

**Fixed:** YOLO26x @1280, conf 0.20, B5 epoch 13, mask 0.45, everything else as C0.

**Cost:** ~1 minute per pass on A100.

**Result:** untested on real data.

**Decision:** pending.

In [ ]:
if RUN_SWEEPS:
    from ultralytics import YOLO

    detector = YOLO(str(YOLO_CHECKPOINT))
    unet, checkpoint = load_checkpoint(UNET_CHECKPOINT, "b5", device)
    sweep_rows = []
    for nms_iou in [0.20, 0.35, 0.50, 0.70]:
        settings = {**DEFAULT_PIPELINE_SETTINGS, "yolo_iou": nms_iou}
        run_id, run_dir = new_run_dir(PROJECT_DIR / "experiments", "pipeline", "C1-2", f"nms_iou_{nms_iou:.2f}")
        by_annotator, summary = evaluate_pipeline(detector, unet, settings, 0.45, run_dir)
        append_registry(registry_path, {"run_id": run_id, "timestamp": datetime.now().isoformat(), "notebook": "04_inference_and_evaluation.ipynb",
                                       "section": "C1.2", "detector": "yolo26x", "detector_imgsz": 1280, "conf": 0.20, "nms_iou": nms_iou,
                                       "end2end": False, "unet_checkpoint": UNET_CHECKPOINT.name, "mask_threshold": 0.45, "tta": False,
                                       "official_pq": summary["official_pq"], "sq": summary["official_sq"], "rq": summary["official_rq"],
                                       "tp": summary["official_tp"], "fp": summary["official_fp"], "fn": summary["official_fn"],
                                       "instances_per_image": summary["instances_per_image"], "pq_ci_low": summary["pq_ci_low"],
                                       "pq_ci_high": summary["pq_ci_high"], "device": str(device), "precision": "fp16" if device.type == "cuda" else "fp32"})
        sweep_rows.append({"run_id": run_id, "nms_iou": nms_iou, **summary})
    display(pd.DataFrame(sweep_rows))

### C1 remaining one-factor sweeps

C1.1 confidence; C1.3 mask threshold; C1.4 dedup; C1.5 minimum mask area; C1.6 crop expansion. Change one key in a copy of `DEFAULT_PIPELINE_SETTINGS`, save every run, and compare it with the same C0 predictions.

### C2 Combine winners

Only combine changes that were useful in the one-factor runs.

### C3 Eight-way U-Net TTA

**Question:** does symmetry averaging improve the B5 stack's generalization?

**Varies:** wrap the loaded U-Net in `RotationTTA`.

**Fixed:** all C0 detector and mask settings.

**Result:** untested on B5.

In [ ]:
if RUN_TTA:
    from ultralytics import YOLO

    detector = YOLO(str(YOLO_CHECKPOINT))
    unet, checkpoint = load_checkpoint(UNET_CHECKPOINT, "b5", device)
    run_id, run_dir = new_run_dir(PROJECT_DIR / "experiments", "pipeline", "C3", "b5_tta")
    by_annotator_tta, summary_tta = evaluate_pipeline(detector, RotationTTA(unet).to(device).eval(), DEFAULT_PIPELINE_SETTINGS.copy(), 0.45, run_dir)
    append_registry(registry_path, {"run_id": run_id, "timestamp": datetime.now().isoformat(), "notebook": "04_inference_and_evaluation.ipynb",
                                   "section": "C3", "detector": "yolo26x", "detector_imgsz": 1280, "conf": 0.20, "nms_iou": 0.20,
                                   "end2end": False, "unet_checkpoint": UNET_CHECKPOINT.name, "mask_threshold": 0.45, "tta": True,
                                   "official_pq": summary_tta["official_pq"], "sq": summary_tta["official_sq"], "rq": summary_tta["official_rq"],
                                   "tp": summary_tta["official_tp"], "fp": summary_tta["official_fp"], "fn": summary_tta["official_fn"],
                                   "instances_per_image": summary_tta["instances_per_image"], "pq_ci_low": summary_tta["pq_ci_low"],
                                   "pq_ci_high": summary_tta["pq_ci_high"], "device": str(device), "precision": "fp16" if device.type == "cuda" else "fp32"})
    display(pd.Series(summary_tta))

### C4 Detector and U-Net ensemble — the current best

**Question:** does agreement across detectors separate real low-confidence filaments from false positives, and does averaging U-Nets reduce Stage 2 variance?

**Varies:** proposals come from weighted box fusion of all seven Stage 1 detectors, each at its training image size and confidence 0.01 (cluster IoU 0.40; a cluster scores the mean of each detector's best box, so missing detectors count as zero; kept at fused score ≥ 0.22). Masks come from the B5 champion averaged with the symmetric-Tversky B5 (`ModelAverage`). Settings live in `ENSEMBLE_*` in `src/config.py`.

**Fixed:** every C0 crop, mask and dedup setting.

**Cost:** ~3 minutes on A100; ~60 minutes on CPU.

**Result (2026-10-27, CPU fp32):** official PQ **0.4628** vs C0 0.4303 on the same path, TP 826 / FP 349 / FN 432 against 789 / 458 / 469, paired delta **+0.0325 [+0.0215, +0.0437]**. Detector fusion alone gave +0.025 and every fused threshold 0.18–0.28 cleared zero; averaging a second U-Net added +0.0077 [+0.0030, +0.0134]; four U-Nets tied two. Kaggle public: see `submissions/leaderboard_log.csv`.

**Decision:** adopted as the submission pipeline in notebook 05.

In [ ]:
if RUN_ENSEMBLE:
    from src.config import ENSEMBLE_SETTINGS, ENSEMBLE_UNETS
    from src.inference import ensemble_proposals
    from src.models import ModelAverage

    validation_paths = [group[0]["image_path"] for group in validation_groups.values()]
    proposals = ensemble_proposals(PROJECT_DIR / "models", validation_paths, device, DEFAULT_PIPELINE_SETTINGS)
    unet = ModelAverage([load_checkpoint(PROJECT_DIR / path, "b5", device)[0] for path in ENSEMBLE_UNETS]).to(device).eval()
    run_id, run_dir = new_run_dir(PROJECT_DIR / "experiments", "pipeline", "C4", "wbf7_unet2")
    by_annotator_ensemble, summary_ensemble = evaluate_pipeline(None, unet, DEFAULT_PIPELINE_SETTINGS.copy(), ENSEMBLE_SETTINGS["mask_threshold"], run_dir, proposals)
    append_registry(registry_path, {"run_id": run_id, "timestamp": datetime.now().isoformat(), "notebook": "04_inference_and_evaluation.ipynb",
                                   "section": "C4", "detector": "wbf7", "detector_imgsz": 1280, "conf": ENSEMBLE_SETTINGS["fused_threshold"], "nms_iou": "",
                                   "end2end": False, "unet_checkpoint": "+".join(ENSEMBLE_UNETS), "mask_threshold": ENSEMBLE_SETTINGS["mask_threshold"], "tta": False,
                                   "other_settings_json": json.dumps(ENSEMBLE_SETTINGS),
                                   "official_pq": summary_ensemble["official_pq"], "sq": summary_ensemble["official_sq"], "rq": summary_ensemble["official_rq"],
                                   "tp": summary_ensemble["official_tp"], "fp": summary_ensemble["official_fp"], "fn": summary_ensemble["official_fn"],
                                   "instances_per_image": summary_ensemble["instances_per_image"], "pq_ci_low": summary_ensemble["pq_ci_low"],
                                   "pq_ci_high": summary_ensemble["pq_ci_high"], "device": str(device), "precision": "fp16" if device.type == "cuda" else "fp32"})
    display(pd.Series(summary_ensemble))
else:
    print("C4 inference skipped. Saved C4 (CPU fp32): PQ 0.4628, TP 826, FP 349, FN 432; paired delta vs C0 +0.0325 [+0.0215, +0.0437].")

### C5 Later experiment

C5 tests a confidence-0.05 fallback only on images where C0 produces zero masks.

### C6 Summary against C0

In [ ]:
from src.metrics import paired_pq_delta

registry = load_registry(registry_path)
display(registry.sort_values("official_pq", ascending=False)[["run_id", "section", "official_pq", "sq", "rq", "tp", "fp", "fn", "kaggle_public", "notes"]])
print("Use paired_pq_delta(control_by_annotator, candidate_by_annotator) when both per-record files are available.")